<a href="https://colab.research.google.com/github/evankim1213/employee-attrition-analytics/blob/main/notebooks/employee_attrition_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.tree import plot_tree

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from xgboost import plot_importance

In [ ]:
df = pd.read_csv('HR_capstone_dataset.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='all')

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
df = df.rename(columns={'average_montly_hours':'average_monthly_hours', 'time_spend_company':'tenure', 'Work_accident':'work_accident', 'Department':'department'})
df.head()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)]

In [ ]:
df['left'].value_counts()

In [ ]:
df['left'].value_counts(normalize=True)

In [ ]:
df1 = df.drop_duplicates(keep='first')
df1.head()

In [ ]:
df1['left'].value_counts()

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns

In [ ]:
df_enc1 = pd.get_dummies(df1, columns=categorical_cols, drop_first=False)
df_enc1.head()

In [ ]:
# Check cross-corrleation matrix to identify potential multicollinearity issues logistic regression
# Drop 'satisfaction_level' because this variable does not exist in real-world HR datasets.

df_enc2 = df_enc1.copy().drop(['satisfaction_level','last_evaluation'], axis=1)


plt.figure(figsize=(12,8))
sns.heatmap(df_enc2.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Cross-Correlation Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(6,6))
sns.boxplot(x=df1['tenure'])
plt.title('Boxplot to detect outliers for tenure')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()

In [ ]:
percentile25 = df1['tenure'].quantile(0.25)
percentile75 = df1['tenure'].quantile(0.75)

iqr = percentile75 - percentile25

lower_limit = percentile25 - 1.5*iqr
upper_limt = percentile75 + 1.5*iqr

print('tenure lower limit:', lower_limit)
print('tenure upper limit:', upper_limt)

outliers = df1[(df1['tenure']<lower_limit) | (df1['tenure']>upper_limt)]
print('# of outliers:', len(outliers))

In [ ]:
df1.describe()

In [ ]:
def plot_boxplot(df, column):
  plt.figure(figsize=(6,6))
  sns.boxplot(x=df[column])
  plt.title(f'Boxplot of {column}')
  plt.xticks(fontsize=12)
  plt.yticks(fontsize=12)
  plt.show()

In [ ]:
plot_boxplot(df1, 'tenure')

In [ ]:
def get_outliers(df, column):
  percentile25 = df[column].quantile(0.25)
  percentile75 = df[column].quantile(0.75)
  iqr = percentile75 - percentile25
  lower_limit = percentile25 - 1.5*iqr
  upper_limit = percentile75 + 1.5*iqr
  outliers = df[(df[column]<lower_limit)|(df[column]>upper_limit)]
  print(f'{column} lower limit:', lower_limit)
  print(f'{column} upper limit:', upper_limit)
  print('# of outliers: ', len(outliers))

In [ ]:
get_outliers(df1,'tenure')

In [ ]:
def plot_multiplots(df, x, y, target):
  fig, ax = plt.subplots(1,2, figsize=(22,8))
  sns.boxplot(data=df, x=x, y=y, hue=target, orient='h', ax=ax[0])
  ax[0].invert_yaxis()
  ax[0].set_title(f'{x} vs {y}', fontsize='14')

  tenure_stay = df[df[target]=='0'][y]
  tenure_left = df[df[target]=='1'][y]
  sns.histplot(data=df, x=y, hue=target, multiple='dodge', shrink=2, ax=ax[1])
  ax[1].set_title(f'{y} histogram', fontsize='14')

  plt.show()

In [ ]:
df1[['number_project','average_monthly_hours']].corr()

In [ ]:
plot_multiplots(df1, 'average_monthly_hours', 'number_project', 'left')

In [ ]:
df1.groupby('number_project')['left'].value_counts(normalize=True)

In [ ]:
#find out whether people leave due to accident
df1.groupby('work_accident')['left'].value_counts(normalize=True)

In [ ]:
#let's deep dive into tenure and salary
#underlying assumptions: people tend to stay longer would have higher salary

fig, ax = plt.subplots(1,3, figsize=(22,12))
long_tenure = df1[df1['tenure'] > 6]
mid_tenure = df1[(df1['tenure'] > 3) & (df1['tenure'] < 7)]
short_tenure = df1[df1['tenure'] < 3]

sns.histplot(data=long_tenure, x='tenure', hue='salary', discrete=1, hue_order=['low','medium','high'], multiple='dodge', shrink=.5, ax=ax[0])
ax[0].set_title('Salary histogram by tenure: long_tenure employees', fontsize=14)


sns.histplot(data=mid_tenure, x='tenure', hue='salary', discrete=1, hue_order=['low','medium','high'], multiple='dodge', shrink=.5, ax=ax[1])
ax[1].set_title('Salary histogram by tenure: mid_tenure employees', fontsize=14)


sns.histplot(data=short_tenure, x='tenure', hue='salary', discrete=0.1, hue_order=['low','medium','high'], multiple='dodge', shrink=.5, ax=ax[2])
ax[2].set_title('Salary histogram by tenure: short_tenure employees', fontsize=14)

plt.show()

In [ ]:
# Analyze how monthly work hour leads to promotion for last 5 years
# People who do not get a reward for hard work would probably expected to leave the company
plt.figure(figsize=(16,3))
sns.scatterplot(data=df1, x='average_monthly_hours', y='promotion_last_5years', hue='left', alpha=0.4)
plt.axvline(x=166.67, color='r', ls='--', label='166.67 hrs/mo')
plt.legend()
plt.title('Monthly hours by promotion last 5 years', fontsize=14)
plt.show()

In [ ]:
#find out more precise point on attriion risk on average_monthly_hours
bins_detail = [80, 120, 160, 200, 240, 260, 270, 280, 300, 320]
df1['hours_bin_detail'] = pd.cut(
    df1['average_monthly_hours'],
    bins=bins_detail,
    include_lowest=True
)

attrition_by_bin_detail = (
    df1
    .groupby('hours_bin_detail')['left']
    .mean()
    .reset_index()
)
print(attrition_by_bin_detail)

In [ ]:
#analyze the pivot point where attrition rate signifcantly increase
plt.plot(attrition_by_bin_detail['hours_bin_detail'].astype(str),
         attrition_by_bin_detail['left'], marker='o')
plt.xticks(rotation=45)
plt.ylabel('Attrition Rate')
plt.xlabel('Monthly Hours Bin')
plt.title('Attrition Rate by Monthly Hours')
plt.show()

In [ ]:
#See if there is any meaningful attrition rate based on departement
plt.figure(figsize=(12,6))
sns.histplot(data=df1, x='department', hue='left', discrete=True, multiple='dodge', shrink=0.5)
plt.xticks(rotation=45)
plt.title('Count of attrition by department', fontsize=14)
plt.show()

In [ ]:
df1.groupby('department')['left'].value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(data=df1, x='department', hue='left', discrete=True, multiple='dodge', shrink=0.5)
dep_rate = df1.groupby('department')['left'].mean().reset_index()
ax1=plt.gca()
ax2=plt.twinx()
sns.lineplot(data=dep_rate, x='department', y='left', ax=ax2, color='r', marker='o', linewidth=2, label='attrition rate')
for dept, rate in zip(dep_rate['department'], dep_rate['left']):
    ax2.text(
        dept,
        rate + 0.001,
        f"{rate*100:.1f}%",
        color='red',
        ha='center',
        va='bottom',
        fontsize=10
    )

ax2.set_ylabel('attrtion rate')
plt.xticks(rotation=45)
plt.title('Count of attrition by department', fontsize=14)
plt.show()

In [ ]:
df1.info()

In [ ]:
#Feature Enginnering
#drop synthetic variables
df2 = df1.copy().drop(['satisfaction_level','last_evaluation','work_accident', 'hours_bin_detail'],axis=1)
df2.head()

In [ ]:
df_enc = df2.copy()

In [ ]:
#encode the categorical variables
df_enc['salary'] = (df_enc['salary']
                    .astype('category')
                    .cat.set_categories(['low','medium','high'])
                    .cat.codes)
df_enc = pd.get_dummies(df_enc, columns = ['department'], drop_first=False)
df_enc.head()

In [ ]:
##synthesize the new variables
#workload_intesity: monthly_average_hours/number_project,
df_enc['workload_intensity'] = df_enc['average_monthly_hours']/df_enc['number_project']

#tenure_bracket: short/mid/long,

df_enc['tenure_bracket'] = pd.cut(df_enc['tenure'],
                                  bins=[0,3,6,100],
                                  labels=[0,1,2])   #short=0, mid=1, long=2

#overwokred_flag: 1 if average_monthly_hours > 280 else 0
df_enc['overworked_flag'] = (df_enc['average_monthly_hours']>270).astype(int)

#promotion_workload_interaction = promotion_last_5years * average_monthly_hours
df_enc['promotion_workload_interaction'] = df_enc['promotion_last_5years'] * df_enc['average_monthly_hours']
df_enc.head()

In [ ]:
##Model buidling
# Baseline model: logistic regression
# (1)split X,y (2) split data into train and test (3) apply standardScaler to train X set

y = df_enc['left']
X = df_enc.copy().drop(['left'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Logistic Regression
#(1)create logisticregression model
logit_reg = LogisticRegression(class_weight='balanced')
#(2)fit the model
logit_reg.fit(X_train_scaled, y_train)
#(3)check the accuracy
y_pred = logit_reg.predict(X_test_scaled)
print(classification_report(y_test,y_pred))

#(4)get coef_ for each features
print(logit_reg.coef_)


In [ ]:
#function to test diffrent threolds
def run_thresholds(model, model_name, X_test, y_test, thresholds):
  # model: model
  # model_name: model_name
  # X_test: X test data to predict
  # y_test: y_test data set
  y_proba = model.predict_proba(X_test)[:,1]
  results = []

  for th in thresholds:
    y_pred_th = (y_proba>=th).astype(int)
    accuracy = accuracy_score(y_test, y_pred_th)
    precision = precision_score(y_test, y_pred_th)
    recall = recall_score(y_test, y_pred_th)
    f1 = f1_score(y_test, y_pred_th)
    auc = np.nan

    results.append([f'{model_name}:{th}', precision, recall, f1, accuracy, auc])
  return results

In [ ]:
def make_threshold_results(results):
  table = pd.DataFrame(results,
                       columns=['model', 'precision', 'recall', 'f1', 'accuracy', 'auc'])
  return table


In [ ]:
thresholds = [0.5, 0.4, 0.3, 0.2]

threshold_result = run_thresholds(logit_reg, 'logit_cls', X_test_scaled, y_test, thresholds)
logit_results = make_threshold_results(threshold_result)
print(logit_results)

In [ ]:
def plot_confusion_matrix(model, y_test, y_pred):
  cm = confusion_matrix(y_test, y_pred)
  cm_disp = ConfusionMatrixDisplay(cm, display_labels = model.classes_)
  cm_disp.plot()
  plt.show()

In [ ]:
plot_confusion_matrix(logit_reg, y_test, y_pred)

In [ ]:
logit_reg.coef_

In [ ]:
logit_coefs = logit_reg.coef_[0]
odds_ratio = np.exp(logit_coefs)

features = X_train.columns

df_or = pd.DataFrame({'feature': features,
                      'coef':logit_coefs,
                      'odds_ratio': odds_ratio}).sort_values('odds_ratio', ascending=False)

In [ ]:
#logistic regression feature importance plot
plt.figure(figsize=(10,6))
plt.barh(df_or['feature'], df_or['odds_ratio'])
plt.gca().invert_yaxis()
plt.title('Odds Ratio by Feature(Logit Regression)')
plt.xlabel('Importance')
plt.show()

In [ ]:
#feature importance plot
def plot_feature_importance(model, feature_names, title:str): ##model should be best model
  importances = model.feature_importances_
  indices = np.argsort(importances)

  plt.figure(figsize=(10,6))
  plt.barh(range(len(importances)), importances[indices])
  plt.yticks(range(len(importances)), feature_names[indices])
  plt.title(title)
  plt.xlabel('Feature Importance')
  plt.tight_layout()
  plt.show()

In [ ]:
#DecisionTree Classifier
dt = DecisionTreeClassifier(random_state=42)

cv_params = {'max_depth':[ 4, 6, 8, None],
             'min_samples_leaf': [1,2,5],
             'min_samples_split': [2,4,6],}
scoring = {'accuracy':'accuracy', 'precision':'precision', 'recall':'recall', 'f1':'f1', 'roc_auc':'roc_auc'}
dt1 = GridSearchCV(dt, cv_params, scoring=scoring, cv=4, refit='recall')

In [ ]:
#Fit deicsion tree to train dataset
%%time
dt1.fit(X_train, y_train)

In [ ]:
dt1.best_params_

In [ ]:
dt1.best_score_

In [ ]:
best_dt1 = dt1.best_estimator_

In [ ]:
y_pred = best_dt1.predict(X_test)

In [ ]:
#review classification report
print(classification_report(y_test, y_pred))

In [ ]:
plot_confusion_matrix(best_dt1, y_test, y_pred)

In [ ]:
#get metrics scores for best model
def get_scores(model_name:str, model, X_test, y_test): #model is before best model chosen
  y_preds = model.best_estimator_.predict(X_test)
  y_probs = model.best_estimator_.predict_proba(X_test)[:,1]
  precision = precision_score(y_test, y_preds)
  recall = recall_score(y_test, y_preds)
  f1 = f1_score(y_test, y_preds)
  accuracy = accuracy_score(y_test, y_preds)
  auc = roc_auc_score(y_test, y_probs)

  table = pd.DataFrame({'model':[model_name],
                       'precision':[precision],
                       'recall':[recall],
                       'accuracy':[accuracy],
                        'f1':[f1],
                       'auc':[auc]})
  return table

In [ ]:
dt_result = get_scores('dt_best1', dt1, X_test, y_test)

In [ ]:
dt_result


In [ ]:
total_result = pd.concat([logit_results, dt_result], axis=0)
total_result

In [ ]:
plot_feature_importance(best_dt1, X_train.columns, 'Feature Importance plot of Decsion Tree')

In [ ]:
#RandomForestClassifier
rf = RandomForestClassifier(random_state=42)

cv_params = {'max_depth': [5,6,7,None],
             'max_features':['sqrt', 0.7],
             'max_samples':[0.7, None],
             'min_samples_leaf': [1,2,3],
             'min_samples_split': [2,3,4],
             'n_estimators':[200, 300]}
scoring = {'precion':'precision', 'recall':'recall','f1':'f1', 'accuracy':'accuracy', 'roc_auc':'roc_auc'}
rf1 = GridSearchCV(rf, cv_params, scoring=scoring, refit='recall', cv=4)

In [ ]:
##fit RF
%%time
rf1.fit(X_train, y_train)

In [ ]:
import pickle
path = '/content/'

In [ ]:
def write_pickle(path, model_object, save_as:str):
  with open(path+save_as+'.pickle', 'wb') as to_write:
    pickle.dump(model_object, to_write)

In [ ]:
def read_pickle(path, saved_model_name:str):
  with open(path+saved_model_name+'.pickle', 'rb') as to_read:
    model = pickle.load(to_read)
  return model

In [ ]:
write_pickle(path, rf1, 'hr_rf1')

In [ ]:
rf1 = read_pickle(path, 'hr_rf1')

In [ ]:
rf1.best_score_

In [ ]:
rf1.best_params_

In [ ]:
y_pred = rf1.predict(X_test)

In [ ]:
get_scores('rf1', rf1, X_test, y_test)

In [ ]:
plot_confusion_matrix(rf1, y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
rf_result = get_scores('rf', rf1, X_test, y_test)
rf_result

In [ ]:
total_result= pd.concat([total_result, rf_result], axis=0)
total_result

In [ ]:
plot_feature_importance(rf1.best_estimator_, X_train.columns, 'Feature Importance of Random Forest')

In [ ]:
#XGBoost Moedl Creating
xgb = XGBClassifier(objective='binary:logistic', random_state=42, tree_method='hist', max_bin=256)
cv_params = {'max_depth': [5,6,7],
             'min_child_weight': [3,5,7],
             'learning_rate':[0.05, 0.1],
             'n_estimators': [300, 500],
             'colsample_bytree': [0.8, 1.0],
             'subsample': [0.8, 1.0]}
scoring = {'precision':'precision', 'recall':'recall', 'accuracy':'accuracy', 'f1':'f1', 'roc_auc':'roc_auc'}
xgb_cv = GridSearchCV(xgb, cv_params, scoring = scoring, cv=4, refit='recall')

In [ ]:
#XGBoost model fit
%%time
xgb_cv.fit(X_train, y_train)

In [ ]:
write_pickle(path, xgb_cv, 'hr_xgb_cv')

In [ ]:
xgb_cv = read_pickle(path, 'hr_xgb_cv')

In [ ]:
xgb_cv.best_estimator_

In [ ]:
xgb_cv.best_score_

In [ ]:
y_pred = xgb_cv.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
xgb_result = get_scores('xgb_cv', xgb_cv, X_test, y_test)
xgb_result

In [ ]:
total_result = pd.concat([total_result, xgb_result], axis=0)
total_result

In [ ]:
plot_confusion_matrix(xgb_cv, y_test, y_pred)

In [ ]:
plot_feature_importance(xgb_cv.best_estimator_, X_train.columns, 'Feature Importance of XGBoosting')

In [ ]:
!pip install shap

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
permutation_result = permutation_importance(
    rf1, X_test, y_test,
    n_repeats=10,
    random_state=42
)

importances = permutation_result.importances_mean
stds = permutation_result.importances_std

for i, v in sorted(zip(X_test.columns, importances), key=lambda x: x[1], reverse=True):
  print(f'{i}: {v:.4f}')

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

In [ ]:
top_features = [
    'number_project',
    'workload_intensity',
    'tenure',
    'tenure_bracket',
    'average_monthly_hours'
]

In [ ]:
#overall direction of top 5 features on partialdependancedisplay
for feature in top_features:
  print(f"PDP for {feature}")
  PartialDependenceDisplay.from_estimator(
      rf1,
      X_test,
      [feature],
      kind='average'
  )
  plt.show()

In [ ]:
type(xgb_cv)

In [ ]:
import shap

xgb_model = xgb_cv.best_estimator_

shap.initjs()
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
#Basline metrcis
actual_attrition_count = y.sum()
actual_attrion_rate = y.mean()*100
employee_population = len(y)

In [ ]:
print(actual_attrion_rate)
print(actual_attrition_count)
print(employee_population)

In [ ]:
##Scenario Analysis - Scenario A: Project Load Optimization
##Evaluates the potential retention impact of reducing excessive project allocation among employees allocated with six or more projects
#baseline calcuation

baseline_proba = rf1.predict_proba(X)[:, 1]
baseline_expected_attrition = baseline_proba.sum()
print(f'Baseline Expected Attrition {baseline_expected_attrition:.2f}')

In [ ]:
#Scenario A
scenario_a = X.copy()

#assume number of project has been rebalanced
scenario_a.loc[scenario_a['number_project'] >=6, 'number_project' ] = 5

scenario_a_proba = rf1.predict_proba(scenario_a)[:,1]
scenario_a_expected_attrition = scenario_a_proba.sum()
scenario_a_attrition_rate = (scenario_a_expected_attrition / len(X)) *100

employees_retained = (baseline_expected_attrition - scenario_a_expected_attrition)
print(f'Scenario A Attrion Count: {scenario_a_expected_attrition:.2f}')
print(f'Scenario A Attrion Rate: {scenario_a_attrition_rate:.2f}%')
print(f'Estimated Employees Retained: {round(employees_retained)}')

In [ ]:
#Scenario B
scn_b = X.copy()
hrs = 'average_monthly_hours'
scn_b.loc[scn_b[hrs] >= 220, hrs] = 200

In [ ]:
proba_b = rf1.predict_proba(scn_b)[:,1]
cnt_b = proba_b.sum()

rate_b = (cnt_b/len(X))*100

ret_b = baseline_expected_attrition - cnt_b
print(f'Scenario B Attrition Count: {cnt_b:.2f}')
print(f'Scenario B Attrition Rate: {rate_b:.2f}%')
print(f'Estimated Employees Retained: {round(ret_b)}')

In [ ]:
(X['average_monthly_hours'] >= 220).sum()

In [ ]:
#Scenario C
scn_c = X.copy()
#Project load optimization
scn_c.loc[scn_c['number_project'] >= 6, 'number_project'] = 5
#Workoing hours optimization
scn_c.loc[scn_c['average_monthly_hours']>=220, 'average_monthly_hours'] = 200

In [ ]:
proba_c = rf1.predict_proba(scn_c)[:,1]

cnt_c = proba_c.sum()
rate_c = (cnt_c/len(X))*100
ret_c = round(baseline_expected_attrition - cnt_c)
print(f'Scenario C Attrition Count: {cnt_c:.2f}')
print(f'Scenario C Attrition Rate: {rate_c:.2f}%')
print(f'Estimated Employees Retained: {ret_c}')

In [ ]:
scenarios = [
    "Baseline",
    "Scenario A",
    "Scenario B",
    "Scenario C"
]

rates = [
    16.63,
    15.60,
    10.41,
    9.27
]

colors = [
    "gray",  # Baseline
    "lightblue",
    "lightblue",
    "darkblue"
]

plt.figure(figsize=(8,5))

bars = plt.bar(
    scenarios,
    rates,
    color=colors,
    width=0.5
)

plt.ylabel("Predicted Attrition Rate (%)")
plt.title("Predicted Attrition Rate by Scenario")

for bar, rate in zip(bars, rates):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        rate + 0.2,
        f"{rate:.2f}%",
        ha="center"
    )

plt.ylim(0, 18)
plt.tight_layout()

plt.savefig(
    "scenario_analysis_attrition_rate.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()